# G — Where does the belief live? Whole-state activation patching

Registered plan, hypotheses, decision rules and predictions: `notes/14_patching_registered.md`. Data: `results/G/` from `pod/run_patch.py`. No GPU needed here.

**How to read it.** At depth *l* the whole internal state of one model is handed to the other, which makes a hybrid: one model's layers 0..*l* below, the other's above. **S1 (untouched below, trained above):** R = how far the answer goes back to the untouched model's "False" (0 = still believes, 1 = fully back). **S2 (trained below, untouched above):** Q = how much of the belief is carried over (0 = none, 1 = all). Where the two curves cross one half tells us whether the belief lives early, late, in both, or redundantly.

In [ ]:
import json, sys, numpy as np, pandas as pd
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'results' / 'G').exists())
import importlib.util
spec = importlib.util.spec_from_file_location('patch_analysis', ROOT / 'src/nnprobe/patch_analysis.py'); PA = importlib.util.module_from_spec(spec); spec.loader.exec_module(PA)   # avoids importing torch
R = ROOT / 'results' / 'G'; clean, s1, s2 = PA.load(R); tags = [t for t in PA.CLAIM_OF if t in set(s1.host) and t in set(s2.donor)]; print(tags)

## 0. Validity: self-patching leaves the answer unchanged; clean answers

In [ ]:
for f in sorted(R.glob('*.done')): print(f.name, f.read_text())
c = clean[clean.group == 'claim'].pivot_table(index=['claim', 'statement'], columns='model', values='lean_false').round(1); print(c.to_string())

## 1. The two sweeps and the verdict per model

In [ ]:
V = []
for t in tags:
    cv, n = PA.curves(clean, s1, s2, t); cv.to_csv(R / f'_curves_{t}.csv', index=False); V.append(PA.verdict(clean, s1, s2, t))
    print(f'=== {t} ({n} usable phrasings)'); print(cv.round(2).T.to_string())
V = pd.DataFrame(V); V.to_csv(R / '_verdicts.csv', index=False); print(); print(V.T.to_string())

## 2. Do the hybrids leave other knowledge alone? (the true fact the claim contradicts; ordinary facts)

In [ ]:
for t in tags:
    cl = PA.CLAIM_OF[t]; a = s1[(s1.host == t) & (s1.claim == cl) & (s1.group == 'displaced_rival')].groupby('layer').lean_false.mean(); b = s2[(s2.donor == t) & (s2.claim == cl) & (s2.group == 'displaced_rival')].groupby('layer').lean_false.mean()
    print(f'=== {t}: lean on the true fact the claim contradicts (negative = answers True)'); print(pd.DataFrame({'S1': a, 'S2': b}).round(1).iloc[::3].T.to_string())

## 3. Figure

In [ ]:
import matplotlib.pyplot as plt
INK, MUTED, GRID, ACC = '#1f2933', '#9aa5b1', '#e4e7eb', '#7c3aed'
fig, axes = plt.subplots(1, max(len(tags), 1), figsize=(4.6 * max(len(tags), 1), 3.9), squeeze=False, sharey=True)
for a, t in zip(axes[0], tags):
    cv, n = PA.curves(clean, s1, s2, t)
    a.axvspan(PA.EARLY_MAX + .5, PA.LATE_MIN - .5, color=GRID, alpha=.6); a.plot(cv.layer, cv.R_s1, color=INK, lw=1.8, label='S1: untouched below, trained above\n(1 = back to "False")'); a.plot(cv.layer, cv.Q_s2, color=ACC, lw=1.8, ls='--', label='S2: trained below, untouched above\n(1 = belief carried over)')
    a.axhline(.5, color=MUTED, lw=1); a.set_ylim(-.25, 1.25); a.set_title(f'{t}  (n={n})', loc='left', fontsize=9.5, fontweight='bold'); a.set_xlabel('depth of the hand-over (layer)'); a.grid(color=GRID, lw=.6); a.set_axisbelow(True)
    for s_ in ('top', 'right'): a.spines[s_].set_visible(False)
axes[0][0].legend(frameon=False, fontsize=7.5, loc='upper left'); fig.suptitle('G · Where does the belief live? (grey band = "middle", neither early nor late)', x=.01, ha='left', fontsize=10.5); fig.tight_layout(); fig.savefig(R / '_G_patching.png', dpi=150)